In [8]:
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
## make plots bigger
from pylab import rcParams
rcParams['figure.figsize'] = 15, 7

# metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# models
from tbats import TBATS
from prophet import Prophet
from pmdarima import auto_arima
from sklearn.preprocessing import StandardScaler

def plot_results(y_to_train,
                 y_to_test, y_forecast,
                 plot_conf_int=True,
                 left_bound=None, right_bound=None):

    plt.plot(y_to_train, label='train')
    plt.plot(y_to_test, label='test')
    plt.plot(y_to_test.index, y_forecast, label='prediction')

    if plot_conf_int:
        plt.fill_between(y_to_test.index,
                         left_bound, right_bound,
                         alpha=0.23, color='grey',
                         label='intervals')
    plt.legend()
    plt.show()

from statsmodels.tsa.statespace.sarimax import SARIMAX

In [2]:
data = pd.read_csv('train.csv', parse_dates=['date'])
data.head()

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [13]:
store, item = 1,1
data = data[(data['store'] == store) & (data['item'] == item)]
data = data.set_index('date')
data = data['sales']

In [14]:
test_size = 365
data_train = data.iloc[: -test_size]
data_test  = data.iloc[-test_size:]

In [15]:
# MAPE
def mean_absolute_percentage_error(y_true, y_pred) -> float:
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [16]:
def add_method_comparison(method: str, y_true, y_forecast, compare_table=None) -> pd.DataFrame:
    result_row = {
        'method': method,
        "MSE": mean_squared_error(y_true=y_true, y_pred=y_forecast),
        "MAE": mean_absolute_error(y_true=y_true, y_pred=y_forecast),
        "MAPE": mean_absolute_percentage_error(y_true=y_true, y_pred=y_forecast)
    }

    if compare_table is None:
        compare_table = pd.DataFrame([result_row])
    else:
        if method in list(compare_table['method']):
            compare_table = compare_table[compare_table['method'] != method]

        compare_table = pd.concat([compare_table, pd.DataFrame([result_row])])
        compare_table.index = np.arange(len(compare_table))
    return compare_table

In [17]:
data

date
2013-01-01    13
2013-01-02    11
2013-01-03    14
2013-01-04    13
2013-01-05    10
              ..
2017-12-27    14
2017-12-28    19
2017-12-29    15
2017-12-30    27
2017-12-31    23
Name: sales, Length: 1826, dtype: int64

In [19]:
# Создание экзогенных переменных
exog = pd.DataFrame({'date': data.index})
exog = exog.set_index(exog['date'])

exog['sin365'] = np.sin(2 * np.pi * exog.index.dayofyear / 365.25)
exog['cos365'] = np.cos(2 * np.pi * exog.index.dayofyear / 365.25)

exog['sin365_2'] = np.sin(4 * np.pi * exog.index.dayofyear / 365.25)
exog['cos365_2'] = np.cos(4 * np.pi * exog.index.dayofyear / 365.25)

# Недельная сезонность
exog['sin7'] = np.sin(2 * np.pi * exog.index.dayofweek / 7)
exog['cos7'] = np.cos(2 * np.pi * exog.index.dayofweek / 7)

# Ежемесячная сезонность
exog['sin30'] = np.sin(2 * np.pi * exog.index.day / 30)
exog['cos30'] = np.cos(2 * np.pi * exog.index.day / 30)

# Добавление дополнительных экзогенных переменных
exog['weekday'] = exog.index.dayofweek

# Преобразование экзогенных переменных
scaler = StandardScaler()
exog[['sin365', 'cos365', 'sin7', 'cos7', 'sin30', 'cos30']] = scaler.fit_transform(exog[['sin365', 'cos365', 'sin7', 'cos7', 'sin30', 'cos30']])

exog = exog.drop(columns=['date'])

# Разделение на тренировочный и тестовый наборы
exog_to_train = exog.iloc[:-test_size]
exog_to_test = exog.iloc[-test_size:]

# Обучение модели SARIMAX с экзогенными переменными
sarimax_model_exog = SARIMAX(data_train, exog=exog_to_train, order=(3, 1, 5), seasonal_order=(0, 0, 2, 7),
                         seasonal_periods=7).fit()

# Прогнозирование с экзогенными переменными
forecast = sarimax_model_exog.get_forecast(steps=len(data_test), exog=exog_to_test)
y_sarimax_forecast_exog = forecast.predicted_mean
conf_int = forecast.conf_int()

C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['seasonal_periods']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Use

In [22]:
compare_table = add_method_comparison('SARIMAX с exog', data_test, y_sarimax_forecast_exog)
compare_table

,method,MSE,MAE,MAPE
0,SARIMAX с exog,24.47767,3.896864,19.522031


In [23]:
exog = pd.DataFrame({'date': data.index})
exog = exog.set_index(exog['date'])

exog['sin365'] = np.sin(2 * np.pi * exog.index.dayofyear / 365.25)
exog['cos365'] = np.cos(2 * np.pi * exog.index.dayofyear / 365.25)

exog['sin365_2'] = np.sin(4 * np.pi * exog.index.dayofyear / 365.25)
exog['cos365_2'] = np.cos(4 * np.pi * exog.index.dayofyear / 365.25)

exog = exog.drop(columns=['date'])

exog_to_train = exog.iloc[:-test_size]
exog_to_test = exog.iloc[-test_size:]

# Обучение модели SARIMAX с экзогенными переменными
sarimax_model_exog_ = SARIMAX(data_train, exog=exog_to_train, order=(3, 1, 5), seasonal_order=(0, 0, 2, 7),
                         seasonal_periods=7).fit()

# Прогнозирование с экзогенными переменными
forecast = sarimax_model_exog_.get_forecast(steps=len(data_test), exog=exog_to_test)
y_sarimax_forecast_exog_ = forecast.predicted_mean
conf_int = forecast.conf_int()

C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['seasonal_periods']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
C:\Users\User\AppData\Roaming\Python\Python310\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Use

In [24]:
compare_table = add_method_comparison('SARIMAX с exog', data_test, y_sarimax_forecast_exog_, compare_table)
compare_table

,method,MSE,MAE,MAPE
0,SARIMAX с exog,32.621035,4.506237,23.676356
